# 4. 1D CNN next-day prediction

Predict **entire next day** using a **1D CNN**: for each (hour, frequency, threshold, class), use the last **seq_len days** of au_pct as input and predict the next day's au_pct.

- **Error:** same as notebook 2 — computed over the entire day → **one MAE and one RMSE per day**; same testing data and error reporting structure.
- **Final visualization:** dropdown for **class**; show testing MAE/RMSE per day. **Final results table:** MAE (1D CNN) per class.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
# Path to final data (same as notebook 2)
work_dir = None
for candidate in [Path("organized"), Path("../organized"), Path("work_dir"), Path("../work_dir")]:
    fd = candidate / "final"
    if fd.exists():
        work_dir = candidate
        break
if work_dir is None:
    raise FileNotFoundError('Final dir not found. Tried: organized/final, ../organized/final, work_dir/final, ../work_dir/final')
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"

In [3]:
def load_final_parquet(split_dir: Path, band: str, date_yyyymmdd: str) -> Optional[pd.DataFrame]:
    """Load final_<date>.parquet for a given split and band. Returns None if missing."""
    path = split_dir / band / f"final_{date_yyyymmdd}.parquet"
    if not path.exists():
        return None
    return pd.read_parquet(path)


def get_dates_and_bands(split_dir: Path):
    """Return sorted list of date_yyyymmdd and list of unique bands."""
    dates = set()
    bands = []
    for band_dir in sorted(split_dir.iterdir()):
        if not band_dir.is_dir():
            continue
        if band_dir.name not in bands:
            bands.append(band_dir.name)
        for p in band_dir.glob("final_*.parquet"):
            d = p.stem.replace("final_", "")
            dates.add(d)
    return sorted(dates), sorted(set(bands))

In [4]:
# Discover bands and dates (same as notebook 2)
training_dates, bands_training = get_dates_and_bands(training_dir)
testing_dates, bands_testing = get_dates_and_bands(testing_dir)
class_options = sorted(set(bands_training) | set(bands_testing))

### 1D CNN hyperparameters

In [5]:
# Sequence length: number of past days used to predict next day
SEQ_LEN = 14
# 1D CNN architecture
FILTERS = [32, 64]
KERNEL_SIZE = 3
DROPOUT = 0.2
# Training
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 1e-3

### Load day-series and build sequences

For each (hour, freq_center_ghz, threshold_dbm) we have a time series of au_pct by date. We use the last `SEQ_LEN` days to predict the next day's au_pct. One 1D CNN per band (shared across all keys).

In [6]:
def load_date_from_train_or_test(band: str, date_yyyymmdd: str) -> Optional[pd.DataFrame]:
    """Load a single day's data from training_dir or testing_dir."""
    df = load_final_parquet(training_dir, band, date_yyyymmdd)
    if df is None:
        df = load_final_parquet(testing_dir, band, date_yyyymmdd)
    return df


def build_day_series_for_band(band: str, dates: list[str]) -> tuple[pd.DataFrame, list[tuple]]:
    """
    Load all dates for band and build a panel: rows = (date, hour, freq_center_ghz, threshold_dbm), col = au_pct.
    Returns (panel with date index and multi-index columns for (hour, freq, thresh), key_cols list).
    Actually we return a dict: date -> df with columns hour, freq_center_ghz, threshold_dbm, au_pct.
    And we need the list of (hour, freq, threshold) keys in stable order.
    """
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    by_date = {}
    keys_set = set()
    for d in dates:
        df = load_date_from_train_or_test(band, d)
        if df is None or df.empty:
            continue
        by_date[d] = df[key_cols + ["au_pct"]].copy()
        for _, row in df[key_cols].drop_duplicates().iterrows():
            keys_set.add((int(row["hour"]), float(row["freq_center_ghz"]), int(row["threshold_dbm"])))
    keys_list = sorted(keys_set)
    return by_date, keys_list


def build_train_sequences(band: str, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    """Build X (n_samples, seq_len, 1) and y (n_samples,) from training dates."""
    by_date, keys_list = build_day_series_for_band(band, training_dates)
    if len(by_date) < seq_len + 1:
        return np.array([]).reshape(0, seq_len, 1), np.array([])
    ordered_dates = sorted(by_date.keys())
    X_list, y_list = [], []
    for (hour, freq, thresh) in keys_list:
        series = []
        for d in ordered_dates:
            df = by_date[d]
            row = df[(df["hour"] == hour) & (df["freq_center_ghz"] == freq) & (df["threshold_dbm"] == thresh)]
            if row.empty:
                series.append(np.nan)
            else:
                series.append(float(row["au_pct"].iloc[0]))
        series = np.array(series, dtype=np.float64)
        if np.isnan(series).any():
            series = pd.Series(series).ffill().bfill().values
        for i in range(seq_len, len(series)):
            X_list.append(series[i - seq_len : i].reshape(-1, 1))
            y_list.append(series[i])
    if not X_list:
        return np.array([]).reshape(0, seq_len, 1), np.array([])
    return np.array(X_list), np.array(y_list)

In [7]:
def build_1d_cnn_model(seq_len: int):
    model = keras.Sequential()
    model.add(layers.Input(shape=(seq_len, 1)))
    for i, f in enumerate(FILTERS):
        model.add(layers.Conv1D(f, KERNEL_SIZE, activation="relu", padding="same"))
        model.add(layers.MaxPooling1D(2))
        model.add(layers.Dropout(DROPOUT))
    model.add(layers.Flatten())
    model.add(layers.Dense(32, activation="relu"))
    model.add(layers.Dropout(DROPOUT))
    model.add(layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse", metrics=["mae"])
    return model

1D CNN: Conv1D layers with MaxPooling and dropout; output is next-day au_pct per sequence.

### Predict next day for a given date

For each (hour, freq, threshold) we need the last SEQ_LEN days ending the day before the target. We use data from training and/or testing in chronological order.

In [8]:
def get_ordered_dates_up_to(target_date_yyyymmdd: str) -> list[str]:
    """All dates (training + testing) up to and including the day before target."""
    from datetime import datetime, timedelta
    try:
        dt = datetime.strptime(target_date_yyyymmdd, "%Y%m%d")
        prev = (dt - timedelta(days=1)).strftime("%Y%m%d")
    except Exception:
        return []
    all_dates = sorted(set(training_dates) | set(testing_dates))
    return [d for d in all_dates if d <= prev]


def predict_one_day(model: keras.Model, band: str, target_date: str, keys_list: list, by_date_cache: dict, seq_len: int) -> Optional[pd.DataFrame]:
    """
    Predict au_pct for target_date for all (hour, freq, threshold) in keys_list.
    seq_len: number of past days (must match model input shape).
    """
    ordered = get_ordered_dates_up_to(target_date)
    if len(ordered) < seq_len:
        return None
    use_dates = ordered[-seq_len:]
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    rows = []
    for (hour, freq, thresh) in keys_list:
        series = []
        for d in use_dates:
            if d not in by_date_cache:
                df = load_date_from_train_or_test(band, d)
                by_date_cache[d] = df[key_cols + ["au_pct"]].copy() if df is not None else None
            df = by_date_cache.get(d)
            if df is None or df.empty:
                series.append(np.nan)
                continue
            row = df[(df["hour"] == hour) & (df["freq_center_ghz"] == freq) & (df["threshold_dbm"] == thresh)]
            if row.empty:
                series.append(np.nan)
            else:
                series.append(float(row["au_pct"].iloc[0]))
        series = np.array(series, dtype=np.float64)
        if np.isnan(series).any():
            series = pd.Series(series).ffill().bfill().values
        X = series.reshape(1, seq_len, 1)
        pred = model.predict(X, verbose=0)[0, 0]
        rows.append({"hour": hour, "freq_center_ghz": freq, "threshold_dbm": thresh, "au_pct": float(pred)})
    return pd.DataFrame(rows)

In [9]:
def compute_cnn_errors_one_band(band: str, model: keras.Model, keys_list: list, by_date_cache: dict, seq_len: int) -> tuple[pd.DataFrame, float, float]:
    """
    For each testing date, predict and compute MAE/RMSE over the day. Also return overall testing MAE and RMSE.
    Returns (testing_per_day DataFrame, testing_MAE, testing_RMSE).
    """
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    per_day_rows = []
    all_errs = []
    for date_curr in testing_dates:
        df_curr = load_final_parquet(testing_dir, band, date_curr)
        if df_curr is None or df_curr.empty:
            continue
        df_pred = predict_one_day(model, band, date_curr, keys_list, by_date_cache, seq_len)
        if df_pred is None or df_pred.empty:
            continue
        df_pred = df_pred.rename(columns={"au_pct": "au_pct_pred"})
        merge = df_curr.merge(df_pred, on=key_cols, how="inner")
        if merge.empty:
            continue
        err = merge["au_pct"] - merge["au_pct_pred"]
        mae = float(np.abs(err).mean())
        rmse = float(np.sqrt((err ** 2).mean()))
        per_day_rows.append({"date_yyyymmdd": date_curr, "MAE": mae, "RMSE": rmse})
        all_errs.extend(err.tolist())
    test_mae = float(np.abs(np.array(all_errs)).mean()) if all_errs else np.nan
    test_rmse = float(np.sqrt((np.array(all_errs) ** 2).mean())) if all_errs else np.nan
    per_day = pd.DataFrame(per_day_rows) if per_day_rows else pd.DataFrame(columns=["date_yyyymmdd", "MAE", "RMSE"])
    return per_day, test_mae, test_rmse

### Train 1D CNN per band and evaluate on testing (same structure as notebook 2)

In [10]:
# Use effective sequence length when we have fewer training days than SEQ_LEN
effective_seq_len = min(SEQ_LEN, len(training_dates) - 1)
if effective_seq_len < 1:
    raise ValueError(f"Need at least 2 training dates; got {len(training_dates)}")
print(f"Training dates: {len(training_dates)}, effective_seq_len: {effective_seq_len}")

results = {}
models = {}
keys_per_band = {}
model_seq_len = {}

for band in class_options:
    X_train, y_train = build_train_sequences(band, effective_seq_len)
    if X_train.size == 0:
        results[band] = {"testing_per_day": pd.DataFrame(), "testing_MAE": np.nan, "testing_RMSE": np.nan}
        continue
    _, keys_per_band[band] = build_day_series_for_band(band, training_dates)
    model = build_1d_cnn_model(effective_seq_len)
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
    models[band] = model
    model_seq_len[band] = effective_seq_len
    cache = {}
    testing_per_day, test_mae, test_rmse = compute_cnn_errors_one_band(band, model, keys_per_band[band], cache, effective_seq_len)
    results[band] = {
        "testing_per_day": testing_per_day,
        "testing_MAE": test_mae,
        "testing_RMSE": test_rmse,
    }

# Cumulative table: Class, MAE, RMSE (same as naive notebook)
final_results = pd.DataFrame([
    {"Class": band, "MAE": results[band]["testing_MAE"], "RMSE": results[band]["testing_RMSE"]}
    for band in class_options
])
print("1D CNN next-day prediction — Final results (testing data)")
display(final_results.round(4))



Training dates: 6, effective_seq_len: 5
Epoch 1/30


2026-02-10 02:15:36.835825: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-02-10 02:15:36.835849: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-02-10 02:15:36.835856: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
I0000 00:00:1770707736.835868 44165519 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1770707736.835884 44165519 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-02-10 02:15:37.318787: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 331.3931 - mae: 7.1999
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 149.6814 - mae: 5.1134
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 124.6670 - mae: 4.4904
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 93.0797 - mae: 3.8556
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 95.8168 - mae: 3.9644 
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 82.3483 - mae: 3.5709
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 84.7743 - mae: 3.6174
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 76.3431 - mae: 3.5077
Epoch 9/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 57.6147 - mae: 2.9789
Epoch 10/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 61.6179 - mae: 3.1150
Epoch 11/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 69.7780 - mae: 3.1712
Epoch 12/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 59.3676 - mae: 3.0412
Epoch 13/30
16/16 ━━━━━━━━━━━━━━━━━━

,Class,MAE,RMSE
0,195MHz,4.0393,9.1570
1,2441MHz,8.7344,11.3659
2,3765MHz,11.7933,16.0989
3,539MHz,5.0564,11.4251
4,5500MHz,3.2170,5.9576
5,915MHz,6.0334,8.5092


In [11]:
# Re-display final results table (optional; same table as naive: Class, MAE, RMSE)
try:
    print("1D CNN next-day prediction — Final results (testing data)")
    display(final_results.round(4))
except NameError:
    print("Run the training cell above first.")

1D CNN next-day prediction — Final results (testing data)


,Class,MAE,RMSE
0,195MHz,4.0393,9.1570
1,2441MHz,8.7344,11.3659
2,3765MHz,11.7933,16.0989
3,539MHz,5.0564,11.4251
4,5500MHz,3.2170,5.9576
5,915MHz,6.0334,8.5092


### Per-class view: dropdown for class — testing MAE/RMSE per day

In [12]:
class_dropdown = widgets.Dropdown(
    options=class_options,
    value=class_options[0] if class_options else None,
    description="Class:",
    style={"description_width": "50px"},
)
out = widgets.Output()


def update_cnn_viz(class_band):
    with out:
        clear_output(wait=True)
        if class_band not in results:
            print(f"No results for class {class_band}")
            return
        r = results[class_band]
        test_df = r["testing_per_day"]
        test_mae = r["testing_MAE"]

        if not test_df.empty:
            test_display = test_df.copy()
            test_display["date"] = test_display["date_yyyymmdd"].str[:4] + "-" + test_display["date_yyyymmdd"].str[4:6] + "-" + test_display["date_yyyymmdd"].str[6:8]
            print(f"1D CNN next-day prediction — Class: {class_band} (testing only)")
            display(test_display[["date", "MAE", "RMSE"]].round(4))
        print(f"Testing MAE (all testing days): {test_mae:.4f}")

        fig = go.Figure()
        if not test_df.empty:
            test_dates_dash = [f"{d[:4]}-{d[4:6]}-{d[6:8]}" for d in test_df["date_yyyymmdd"]]
            fig.add_trace(go.Bar(x=test_dates_dash, y=test_df["MAE"], name="MAE (per day)", marker_color="seagreen"))
        fig.update_layout(
            title=f"1D CNN next-day MAE by date — {class_band}",
            xaxis_title="Date",
            yaxis_title="MAE",
            height=400,
        )
        fig.show()


widgets.interactive_output(update_cnn_viz, {"class_band": class_dropdown})
display(widgets.HBox([class_dropdown]), out)
update_cnn_viz(class_dropdown.value)

Output()

### Predicted vs Actual heatmaps

Pick a **class** and **testing date**. Left: actual AU (%) for that day. Right: 1D CNN prediction (same day). Same colorscale for comparison.